# 02 Data Cleaning

This notebook documents the final cleaning steps and exports clean datasets for the rest of the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# File paths
orders_path = '/mnt/data/orders_cl.csv'
products_path = '/mnt/data/products_cl(1).csv'
orderlines_path = '/mnt/data/orderlines_qu.csv'
brands_path = '/mnt/data/brands(1).csv'

# Load datasets
orders_cl = pd.read_csv(orders_path)
products_cl = pd.read_csv(products_path)
orderlines_qu = pd.read_csv(orderlines_path)
brands_df = pd.read_csv(brands_path)

print('orders_cl:', orders_cl.shape)
print('products_cl:', products_cl.shape)
print('orderlines_qu:', orderlines_qu.shape)
print('brands_df:', brands_df.shape)


## 1. Clean duplicates

In [ ]:
orders_clean = orders_cl.drop_duplicates().copy()
products_clean = products_cl.drop_duplicates().copy()
orderlines_clean = orderlines_qu.drop_duplicates().copy()
brands_clean = brands_df.drop_duplicates().copy()

# Keep SKU unique in products if needed
products_clean = products_clean.drop_duplicates(subset='sku')

print('orders_clean:', orders_clean.shape)
print('products_clean:', products_clean.shape)
print('orderlines_clean:', orderlines_clean.shape)
print('brands_clean:', brands_clean.shape)

## 2. Convert data types

In [ ]:
orders_clean['created_date'] = pd.to_datetime(orders_clean['created_date'], errors='coerce')
orderlines_clean['date'] = pd.to_datetime(orderlines_clean['date'], errors='coerce')

# Numeric safety
orders_clean['total_paid'] = pd.to_numeric(orders_clean['total_paid'], errors='coerce')
products_clean['price'] = pd.to_numeric(products_clean['price'], errors='coerce')
orderlines_clean['unit_price'] = pd.to_numeric(orderlines_clean['unit_price'], errors='coerce')
orderlines_clean['product_quantity'] = pd.to_numeric(orderlines_clean['product_quantity'], errors='coerce')
products_clean['in_stock'] = pd.to_numeric(products_clean['in_stock'], errors='coerce')

print(orders_clean.dtypes)
print(products_clean.dtypes)
print(orderlines_clean.dtypes)

## 3. Remove impossible or unusable values

In [ ]:
# Missing critical values
orders_clean = orders_clean.dropna(subset=['order_id', 'created_date', 'total_paid', 'state'])
products_clean = products_clean.dropna(subset=['sku', 'name', 'price'])
orderlines_clean = orderlines_clean.dropna(subset=['id_order', 'sku', 'unit_price', 'product_quantity', 'date'])

# Keep only positive prices and quantities
products_clean = products_clean[products_clean['price'] > 0]
orderlines_clean = orderlines_clean[(orderlines_clean['unit_price'] > 0) & (orderlines_clean['product_quantity'] > 0)]

print('orders_clean:', orders_clean.shape)
print('products_clean:', products_clean.shape)
print('orderlines_clean:', orderlines_clean.shape)

## 4. Sanity checks after cleaning

In [ ]:
print('Duplicate SKUs:', products_clean['sku'].duplicated().sum())
print('Missing price values:', products_clean['price'].isna().sum())
print('Missing unit_price values:', orderlines_clean['unit_price'].isna().sum())
print('Missing order dates:', orderlines_clean['date'].isna().sum())

print('\nStates:')
display(orders_clean['state'].value_counts())

## 5. Export clean datasets

In [ ]:
clean_dir = '/mnt/data/eniac_project_notebooks/cleaned_data'
Path(clean_dir).mkdir(parents=True, exist_ok=True)

orders_clean.to_csv(f'{clean_dir}/orders_clean.csv', index=False)
products_clean.to_csv(f'{clean_dir}/products_clean.csv', index=False)
orderlines_clean.to_csv(f'{clean_dir}/orderlines_clean.csv', index=False)
brands_clean.to_csv(f'{clean_dir}/brands_clean.csv', index=False)

print('Clean files exported to:', clean_dir)

## 6. Cleaning decisions

Document what you removed, what you kept, and why.